# Final Product: 2026 Monthly GHI Forecast with the Proposed Model

The final step of the study. The proposed model, **Hybrid SARIMAX-LSTM + CSI + PL**, is retrained on all data up to December 2025 and forecasts the **12 monthly GHI values of 2026** for each of the 93 grid cells. The forecasts are then mapped.

**Nothing about the model changes.** Every setting comes from model development:
- **SARIMAX (CSI) component:** the orders selected during rolling validation.
- **LSTM (CSI + PL) component:** the same architecture, penalized loss, early-stopping rule and random seed.
- **Combination:** the same time-varying weighted average (TVWA, δ = 0.8).

## Steps

| Step | What happens |
|---|---|
| 1. Data | Download NASA POWER monthly GHI, cloud amount and RH for **2005–2025** into a **new file**. The 2005–2024 file behind every reported result is not modified. |
| 2. Validation | The pass/fail checks of the pre-processing step, plus a comparison of 2005–2024 against the original file in case NASA revised past values. |
| 3. Clear-sky | Ineichen-Perez clear-sky GHI (2005–2026, from `Hybrid_Models.ipynb`) and the clear-sky index. |
| 4–5. Components | SARIMAX (CSI) and LSTM (CSI + PL) are fitted twice: on data up to Dec 2024 to forecast **2025**, and on data up to Dec 2025 to forecast **2026**. |
| 6. Combination | TVWA weights from the components' past errors (2014–2025). The **2025 forecast is checked against the observed 2025 data**, an extra out-of-sample year. Its errors also feed the 2026 weights. |
| 7. Maps | Forecast annual mean GHI for 2026, and the 12 monthly forecasts. |

## Outputs
- `outputs/final_product_2026/forecast_2026_monthly.csv`: 93 grids × 12 months, with forecast GHI, clear-sky index, clear-sky GHI, component forecasts and TVWA weight
- `outputs/final_product_2026/forecast_2026_annual.csv`: annual mean daily GHI and annual total per grid, compared with the 2005–2025 average, plus the model's typical monthly error per grid
- `outputs/final_product_2026/check_2025_metrics.csv`: accuracy of the 2025 forecast against observations
- `outputs/figures/forecast_2026_annual_map` and `forecast_2026_monthly_maps` (PNG, SVG, PDF)

## Running it
Run from the `Philippine Spatial Map` folder, with the cells in order. It needs an internet connection for the NASA POWER download (step 1, about 1–2 minutes) and the outputs of `Hybrid_Models.ipynb` and `Final_Testing_Hybrid.ipynb`. Total runtime is about 5–10 minutes.

If NASA POWER hasn't yet published every month of 2025, the validation step stops with a FAIL message. Wait and run again with `REFRESH_EXTRACTION = True`.

In [ ]:
"""
final_product_2026.py

Purpose:
    Retrain the proposed model (Hybrid SARIMAX-LSTM + CSI + PL) on all
    data up to December of the year before the forecast year and
    forecast the 12 monthly GHI values of the forecast year.

Output:
    outputs/final_product_<year>/
    outputs/figures/forecast_<year>_*
"""

from pathlib import Path
from importlib import metadata
import ast
import copy
import json
import platform
import random
import time
import warnings

import numpy as np
import pandas as pd
import requests


# ============================================================
# 1. USER SETTINGS
# ============================================================

FORECAST_YEAR = 2026

FIRST_DATA_YEAR = 2005

# Data are needed up to December of the year before the forecast
LAST_DATA_YEAR = FORECAST_YEAR - 1

# The last data year is also forecast (from the December before it)
# and checked against its observations; its errors feed the TVWA
# weights of the final forecast
CHECK_YEAR = LAST_DATA_YEAR

FORECAST_YEARS = [CHECK_YEAR, FORECAST_YEAR]

# True = download the NASA POWER data again even if the file exists
REFRESH_EXTRACTION = False


# ------------------------------------------------------------
# INPUT FILES
# ------------------------------------------------------------

GRID_FILE = Path("data/processed/grid_metadata.csv")

GRID_GEOJSON = Path("data/processed/philippines_grid.geojson")

BOUNDARY_FILE = Path("data/boundaries/Philippines.shp/PH_Adm0_Country.shp.shp")

ELEVATION_FILE = Path("data/raw/elevation/grid_elevation.csv")

CLEARSKY_FILE = Path("data/processed/clearsky_ineichen_monthly.csv")

# Dataset behind every reported result (2005-2024); never modified here
DEVELOPMENT_FILE = Path("data/raw/nasa_power/nasa_power_monthly.csv")

# New extraction including the latest data year
EXTENDED_FILE = Path(f"data/raw/nasa_power/nasa_power_monthly_{FIRST_DATA_YEAR}_{LAST_DATA_YEAR}.csv")

VALIDATED_FILE = Path(f"data/processed/nasa_power_validated_{FIRST_DATA_YEAR}_{LAST_DATA_YEAR}.csv")

# From model development
SARIMAX_ORDERS_FILE = Path("outputs/hybrid_models/models/sarimax_component_orders.csv")

COMPONENT_HISTORY = {
    "sarimax_csi": [
        Path("outputs/hybrid_models/components/sarimax_csi_rolling.csv"),
        Path("outputs/final_test_hybrid/components/sarimax_csi_final_test.csv")
    ],
    "lstm_csi_pl": [
        Path("outputs/hybrid_models/components/lstm_csi_pl_rolling.csv"),
        Path("outputs/final_test_hybrid/components/lstm_csi_pl_final_test.csv")
    ]
}

# Optional: the proposed model's per-grid RMSE over 2015-2024
PER_GRID_TESTS_FILE = Path("outputs/significance_tests/per_grid_dm_hln.csv")


# ------------------------------------------------------------
# OUTPUT
# ------------------------------------------------------------

OUTPUT_DIR = Path(f"outputs/final_product_{FORECAST_YEAR}")

MODEL_DIR = OUTPUT_DIR / "models"

FIGURE_DIR = Path("outputs/figures")


# ------------------------------------------------------------
# NASA POWER API (same request as the original extraction)
# ------------------------------------------------------------

BASE_URL = "https://power.larc.nasa.gov/api/temporal/monthly/point"

NASA_PARAMETERS = "ALLSKY_SFC_SW_DWN,CLOUD_AMT,RH2M"

COMMUNITY = "RE"

NASA_FILL_VALUE = -999

REQUEST_TIMEOUT = 60

REQUEST_DELAY = 0.2

# Retries for a grid cell, with the wait (seconds) before each new attempt
RETRY_WAITS = [5, 15, 30, 60]

# Stop early if this many grid cells in a row cannot be downloaded
MAX_CONSECUTIVE_FAILURES = 5


# ------------------------------------------------------------
# VARIABLES AND SETTINGS (identical to model development)
# ------------------------------------------------------------

GHI_COL = "ghi_kwh_m2_day"

CLOUD_COL = "cloud_amt_pct"

RH_COL = "rh2m_pct"

CLEAR_COL = "ghi_clear_kwh_m2_day"

CSI_COL = "csi"

VARIABLES = [GHI_COL, CLOUD_COL, RH_COL]

EXOG_COLS = [CLOUD_COL, RH_COL]

STATIC_COLS = ["latitude", "longitude", "elevation_m"]

CALENDAR_COLS = ["month_sin", "month_cos"]

HORIZON = 12

WINDOW = 12

SEASONAL_PERIOD = 12

CSI_MAX = 1.0

SEED = 42

PROPOSED = "Hybrid SARIMAX-LSTM + CSI + PL"

UNITS = "kWh/m²/day"


# ============================================================
# 2. SETUP
# ============================================================

for directory in [OUTPUT_DIR, MODEL_DIR, FIGURE_DIR, EXTENDED_FILE.parent, VALIDATED_FILE.parent]:
    directory.mkdir(parents=True, exist_ok=True)


def set_seed(seed=SEED):
    """Fix the Python and NumPy random seeds."""

    random.seed(seed)

    np.random.seed(seed)


set_seed()

versions = {"python": platform.python_version()}

for package in ["numpy", "pandas", "statsmodels", "scikit-learn", "torch", "pvlib", "joblib", "geopandas", "matplotlib"]:
    try:
        versions[package] = metadata.version(package)
    except metadata.PackageNotFoundError:
        versions[package] = "not installed"

(OUTPUT_DIR / "environment.txt").write_text(
    "\n".join(f"{name}=={version}" for name, version in versions.items()), encoding="utf-8"
)

grid_metadata = pd.read_csv(GRID_FILE)

print(f"Forecast year: {FORECAST_YEAR}   data: {FIRST_DATA_YEAR}-{LAST_DATA_YEAR}   "
      f"check year: {CHECK_YEAR}   grid cells: {len(grid_metadata)}")


# ============================================================
# 3. NASA POWER DATA (FIRST_DATA_YEAR - LAST_DATA_YEAR)
# ============================================================
# Each grid cell is saved to a cache folder as soon as it is downloaded,
# so if the connection drops, running this cell again only downloads
# the missing cells.

import shutil

CACHE_DIR = EXTENDED_FILE.parent / f"nasa_power_cache_{FIRST_DATA_YEAR}_{LAST_DATA_YEAR}"


def request_point(latitude, longitude):
    """One NASA POWER request for one coordinate (raises on any failure)."""

    params = {
        "parameters": NASA_PARAMETERS,
        "community":  COMMUNITY,
        "longitude":  longitude,
        "latitude":   latitude,
        "start":      FIRST_DATA_YEAR,
        "end":        LAST_DATA_YEAR,
        "format":     "JSON"
    }

    response = requests.get(BASE_URL, params=params, timeout=REQUEST_TIMEOUT)

    response.raise_for_status()

    return response.json()


def describe_failure(error):
    """Plain-language cause of a failed request."""

    text = str(error)

    if isinstance(error, requests.exceptions.SSLError) or "CERTIFICATE_VERIFY_FAILED" in text:
        return ("The SSL certificate check failed. Antivirus HTTPS scanning (e.g. Avast Web Shield) "
                "or a network proxy is intercepting the connection. Fix: run '%pip install truststore', "
                "then add 'import truststore; truststore.inject_into_ssl()' at the top of this cell.")

    if "NameResolutionError" in text or "getaddrinfo failed" in text or "Failed to resolve" in text:
        return ("Your computer could not look up the NASA POWER server (DNS failure). Check the internet "
                "connection, try another network (e.g. a phone hotspot) or set your DNS server to 8.8.8.8, "
                "then run this cell again.")

    if isinstance(error, requests.exceptions.HTTPError) and error.response is not None:

        status = error.response.status_code

        try:
            details = error.response.json()
            details = details.get("messages") or details.get("message") or details
        except ValueError:
            details = error.response.text[:400]

        if status == 422:
            return (f"NASA POWER rejected the request (HTTP 422): {details}. This usually means data for "
                    f"{LAST_DATA_YEAR} are not fully published yet - try again later.")

        return f"NASA POWER returned HTTP {status}: {details}"

    if isinstance(error, (requests.exceptions.Timeout, requests.exceptions.ConnectionError)):
        return ("The NASA POWER server did not respond or dropped the connection; it may be busy. "
                "Run this cell again later - downloaded cells are kept.")

    return text


def download_with_retries(row):
    """Download one grid cell, retrying with growing waits. Returns (payload, error)."""

    error = None

    for wait in [0] + RETRY_WAITS:

        time.sleep(wait)

        try:
            return request_point(row.query_latitude, row.query_longitude), None
        except requests.RequestException as caught:
            error = caught

            # A rejected request (HTTP 4xx) will not succeed on retry
            if isinstance(caught, requests.exceptions.HTTPError) and caught.response is not None \
                    and 400 <= caught.response.status_code < 500:
                break

    return None, error


def cache_path(grid_number):
    return CACHE_DIR / f"grid_{grid_number:03d}.json"


def download_all(grid):
    """Download every grid cell that is not cached yet."""

    CACHE_DIR.mkdir(parents=True, exist_ok=True)

    pending = [row for row in grid.itertuples(index=False) if not cache_path(row.grid_number).exists()]

    print(f"{len(grid) - len(pending)} of {len(grid)} grid cells already downloaded; {len(pending)} to go")

    if not pending:
        return

    # Connection check on the first pending cell: stop at once with a clear cause
    first = pending[0]

    try:
        payload = request_point(first.query_latitude, first.query_longitude)
    except requests.RequestException as error:
        raise RuntimeError(
            f"Cannot download from NASA POWER.\nCause: {describe_failure(error)}\nDetails: {error}"
        ) from None

    cache_path(first.grid_number).write_text(json.dumps(payload), encoding="utf-8")

    failed, in_a_row = [], 0

    for position, row in enumerate(pending[1:], start=2):

        payload, error = download_with_retries(row)

        if payload is None:

            failed.append((row.grid_number, describe_failure(error), str(error)))

            in_a_row += 1

            if in_a_row >= MAX_CONSECUTIVE_FAILURES:
                print(f"  Stopping: {in_a_row} grid cells in a row failed.")
                break

            continue

        in_a_row = 0

        cache_path(row.grid_number).write_text(json.dumps(payload), encoding="utf-8")

        if position % 10 == 0 or position == len(pending):
            print(f"  {position}/{len(pending)} downloaded")

        time.sleep(REQUEST_DELAY)

    missing = [row.grid_number for row in grid.itertuples(index=False) if not cache_path(row.grid_number).exists()]

    if missing:
        cause = failed[-1][1] if failed else "unknown"
        details = failed[-1][2] if failed else ""
        raise RuntimeError(
            f"{len(missing)} of {len(grid)} grid cells could not be downloaded: {missing}\n"
            f"Cause: {cause}\nDetails: {details}\n"
            f"Run this cell again to download only the missing cells."
        )


def assemble(grid):
    """Combine the cached downloads into one monthly table."""

    records, header = [], None

    for row in grid.itertuples(index=False):

        payload = json.loads(cache_path(row.grid_number).read_text(encoding="utf-8"))

        header = header or payload.get("header", {})

        parameters = payload["properties"]["parameter"]

        for key, ghi in parameters["ALLSKY_SFC_SW_DWN"].items():

            year, month = int(key[:4]), int(key[4:])

            # Month 13 is the annual value, not a month
            if month == 13:
                continue

            records.append({
                "grid_number": row.grid_number,
                "grid_id":     row.grid_id,
                "latitude":    row.query_latitude,
                "longitude":   row.query_longitude,
                "year":        year,
                "month":       month,
                "date":        pd.Timestamp(year=year, month=month, day=1),
                GHI_COL:       ghi,
                CLOUD_COL:     parameters["CLOUD_AMT"].get(key),
                RH_COL:        parameters["RH2M"].get(key)
            })

    frame = pd.DataFrame(records).sort_values(["grid_number", "date"]).reset_index(drop=True)

    return frame, header


if REFRESH_EXTRACTION and CACHE_DIR.exists():
    shutil.rmtree(CACHE_DIR)

if REFRESH_EXTRACTION or not EXTENDED_FILE.exists():

    print(f"\nDownloading NASA POWER monthly data {FIRST_DATA_YEAR}-{LAST_DATA_YEAR}...")

    download_all(grid_metadata)

    extended, header = assemble(grid_metadata)

    extended.to_csv(EXTENDED_FILE, index=False)

    (OUTPUT_DIR / "nasa_power_request_header.json").write_text(
        json.dumps(header, indent=2, default=str), encoding="utf-8"
    )

    print(f"Saved: {EXTENDED_FILE}")

else:

    print(f"\nUsing existing extraction: {EXTENDED_FILE}")
    print("(set REFRESH_EXTRACTION = True to download again)")

## Validation of the extended dataset

In [ ]:
# ============================================================
# 4. VALIDATE THE EXTENDED DATASET
# ============================================================

data = (
    pd.read_csv(EXTENDED_FILE, parse_dates=["date"])
    .sort_values(["grid_number", "date"])
    .reset_index(drop=True)
)

failures = []


def check(name, passed, detail):

    print(f"[{'PASS' if passed else 'FAIL'}] {name}: {detail}")

    if not passed:
        failures.append(name)


expected_dates = pd.date_range(f"{FIRST_DATA_YEAR}-01-01", f"{LAST_DATA_YEAR}-12-01", freq="MS")

print("\n" + "=" * 65)
print(f"VALIDATION OF {EXTENDED_FILE.name}")
print("=" * 65)


# ------------------------------------------------------------
# Structure
# ------------------------------------------------------------

check("Grid cells match grid_metadata.csv",
      set(data["grid_number"]) == set(grid_metadata["grid_number"]),
      f"{data['grid_number'].nunique()} grids in data, {len(grid_metadata)} in metadata")

months_per_grid = data.groupby("grid_number").size()

check(f"Every grid has {len(expected_dates)} months",
      bool((months_per_grid == len(expected_dates)).all()),
      f"{months_per_grid.min()}-{months_per_grid.max()} months per grid")

duplicates = int(data.duplicated(["grid_number", "date"]).sum())

check("No duplicate grid-month records", duplicates == 0, f"{duplicates} duplicates")

grids_with_gaps = sum(
    not pd.DatetimeIndex(group["date"]).equals(expected_dates)
    for _, group in data.groupby("grid_number")
)

check(f"Continuous {expected_dates[0]:%Y-%m} to {expected_dates[-1]:%Y-%m}",
      grids_with_gaps == 0, f"{grids_with_gaps} grids with gaps or extra months")


# ------------------------------------------------------------
# Missing values (NaN or the NASA POWER fill value)
# ------------------------------------------------------------

for variable in VARIABLES:

    missing = int(data[variable].isna().sum() + (data[variable] == NASA_FILL_VALUE).sum())

    check(f"No missing {variable}", missing == 0, f"{missing} missing or fill values")

data[VARIABLES] = data[VARIABLES].replace(NASA_FILL_VALUE, np.nan)


# ------------------------------------------------------------
# Physical validity (same rules as the pre-processing step)
# ------------------------------------------------------------

SOLAR_CONSTANT = 1361.0


def daily_extraterrestrial_irradiation(latitude_deg, day_of_year):
    """Daily extraterrestrial irradiation, kWh/m²/day (Duffie & Beckman, 2013)."""

    phi = np.radians(latitude_deg)

    e0 = 1 + 0.033 * np.cos(2 * np.pi * day_of_year / 365)

    delta = np.radians(23.45) * np.sin(2 * np.pi * (284 + day_of_year) / 365)

    ws = np.arccos(np.clip(-np.tan(phi) * np.tan(delta), -1, 1))

    return ((24 / np.pi) * SOLAR_CONSTANT * e0
            * (np.cos(phi) * np.cos(delta) * np.sin(ws) + ws * np.sin(phi) * np.sin(delta)) / 1000)


days = pd.date_range(expected_dates[0], expected_dates[-1] + pd.offsets.MonthEnd(0), freq="D")

daily = data[["grid_number", "latitude"]].drop_duplicates().merge(pd.DataFrame({"day": days}), how="cross")

daily["h0_kwh_m2_day"] = daily_extraterrestrial_irradiation(
    daily["latitude"].to_numpy(), daily["day"].dt.dayofyear.to_numpy()
)

daily["date"] = daily["day"].dt.to_period("M").dt.to_timestamp()

data = data.merge(
    daily.groupby(["grid_number", "date"], as_index=False)["h0_kwh_m2_day"].mean(),
    on=["grid_number", "date"], how="left"
)

data["kt"] = data[GHI_COL] / data["h0_kwh_m2_day"]

ghi_invalid = int(((data[GHI_COL] < 0) | (data[GHI_COL] > data["h0_kwh_m2_day"])).sum())

check("GHI within 0 and extraterrestrial H0 (Kt <= 1)", ghi_invalid == 0,
      f"{ghi_invalid} invalid; Kt {data['kt'].min():.3f}-{data['kt'].max():.3f}")

for variable in [CLOUD_COL, RH_COL]:

    outside = int(((data[variable] < 0) | (data[variable] > 100)).sum())

    check(f"{variable} within 0-100 %", outside == 0,
          f"{outside} outside; observed {data[variable].min():.2f}-{data[variable].max():.2f}")


# ------------------------------------------------------------
# Revisions: compare the overlap with the development dataset
# ------------------------------------------------------------

development = pd.read_csv(DEVELOPMENT_FILE, parse_dates=["date"])

overlap = development[["grid_number", "date"] + VARIABLES].merge(
    data[["grid_number", "date"] + VARIABLES],
    on=["grid_number", "date"], suffixes=("_development", "_extended")
)

revisions = []

for variable in VARIABLES:

    change = (overlap[f"{variable}_extended"] - overlap[f"{variable}_development"]).abs()

    revisions.append({
        "variable": variable,
        "values_compared": len(overlap),
        "values_revised": int((change > 1e-6).sum()),
        "max_absolute_change": change.max(),
        "mean_absolute_change": change.mean()
    })

revisions = pd.DataFrame(revisions)

revisions.to_csv(OUTPUT_DIR / "nasa_power_revisions_vs_development.csv", index=False)

print(f"\nOverlap with the development dataset ({development['date'].min():%Y-%m} to "
      f"{development['date'].max():%Y-%m}):")

print(revisions.round(4).to_string(index=False))

if revisions["values_revised"].sum():
    print("NOTE: NASA POWER revised some past values. The final product uses the current values; "
          "the reported results used the development dataset.")


if failures:
    raise RuntimeError(
        f"Validation failed: {failures}. If months of {LAST_DATA_YEAR} are missing, NASA POWER "
        f"has not published them yet - run again later with REFRESH_EXTRACTION = True."
    )


# ------------------------------------------------------------
# Model inputs: elevation and month encoding (as in development)
# ------------------------------------------------------------

elevation = pd.read_csv(ELEVATION_FILE)[["grid_number", "elevation_m"]]

data = data.merge(elevation, on="grid_number", how="left")

data["month_sin"] = np.sin(2 * np.pi * data["month"] / 12)

data["month_cos"] = np.cos(2 * np.pi * data["month"] / 12)

data.to_csv(VALIDATED_FILE, index=False)

print(f"\nAll checks passed. Saved: {VALIDATED_FILE}")

## Clear-sky GHI and clear-sky index

In [ ]:
# ============================================================
# 5. INEICHEN-PEREZ CLEAR-SKY GHI AND CLEAR-SKY INDEX
# ============================================================

if not CLEARSKY_FILE.exists():
    raise FileNotFoundError(f"{CLEARSKY_FILE} not found. Run Hybrid_Models.ipynb first.")

clearsky = pd.read_csv(CLEARSKY_FILE, parse_dates=["date"])

needed = pd.date_range(f"{FIRST_DATA_YEAR}-01-01", f"{FORECAST_YEAR}-12-01", freq="MS")

coverage = clearsky[clearsky["date"].isin(needed)].groupby("grid_number").size()

if len(coverage) != len(grid_metadata) or (coverage < len(needed)).any():
    raise ValueError(
        f"The clear-sky table does not cover {needed[0]:%Y-%m} to {needed[-1]:%Y-%m} for every grid. "
        f"Increase CLEARSKY_END in Hybrid_Models.ipynb, delete {CLEARSKY_FILE} and run its clear-sky cell."
    )

data = data.drop(columns=[CLEAR_COL, CSI_COL], errors="ignore").merge(
    clearsky[["grid_number", "date", CLEAR_COL]], on=["grid_number", "date"], how="left"
)

data[CSI_COL] = data[GHI_COL] / data[CLEAR_COL]

GRID_INFO = (
    data[["grid_number", "grid_id", "latitude", "longitude"]]
    .drop_duplicates("grid_number")
    .reset_index(drop=True)
)

N_GRIDS = len(GRID_INFO)

print(f"CSI {FIRST_DATA_YEAR}-{LAST_DATA_YEAR}: {data[CSI_COL].min():.3f}-{data[CSI_COL].max():.3f} "
      f"(mean {data[CSI_COL].mean():.3f})")

## SARIMAX (CSI) component

Fixed orders from rolling validation. For each forecast year, the model is fitted on data up to December of the year before, warm-started from the previous fit.

In [ ]:
# ============================================================
# 6. SARIMAX (CSI) COMPONENT
# ============================================================

import os

from joblib import Parallel, delayed
from statsmodels.stats.diagnostic import acorr_ljungbox
from statsmodels.tsa.statespace.sarimax import SARIMAX


# Settings identical to model development
EXOG_LAG = 12

SEASONAL_D = 1

MAX_ITER = 200

# Parallel workers. Each worker is kept single-threaded so the run fits in
# 8 GB of RAM; only 93 grids x 2 fits are needed, so 4 workers are enough.
N_JOBS = 4

for variable in ("OPENBLAS_NUM_THREADS", "OMP_NUM_THREADS", "MKL_NUM_THREADS"):
    os.environ[variable] = "1"


order_table = pd.read_csv(SARIMAX_ORDERS_FILE)

order_table = order_table[order_table["target"] == "csi"].set_index("grid_number")

SARIMAX_ORDERS = {
    grid_number: (ast.literal_eval(row["order"]), ast.literal_eval(row["seasonal_order"]))
    for grid_number, row in order_table.iterrows()
}


def fit_sarimax(endog, exog, order, seasonal_order, start_params=None):
    """Fit one SARIMAX model, silencing optimiser warnings."""

    with warnings.catch_warnings():

        warnings.simplefilter("ignore")

        return SARIMAX(endog, exog=exog, order=order, seasonal_order=seasonal_order).fit(
            start_params=start_params, maxiter=MAX_ITER, disp=False
        )


def run_sarimax_grid(grid_number, frame, order, seasonal_order):
    """Fit up to December of the previous year and forecast each year in FORECAST_YEARS."""

    series = frame.set_index("date").asfreq("MS")

    lagged_exog = series[EXOG_COLS].shift(EXOG_LAG)

    endog = series[CSI_COL]

    start = series.index[EXOG_LAG]

    forecasts, diagnostics, result = [], [], None

    for year in FORECAST_YEARS:

        train_end = pd.Timestamp(f"{year - 1}-12-01")

        result = fit_sarimax(
            endog.loc[start:train_end], lagged_exog.loc[start:train_end],
            order, seasonal_order,
            start_params=None if result is None else result.params
        )

        future = pd.date_range(f"{year}-01-01", periods=HORIZON, freq="MS")

        # Cloud amount and RH of the same months one year earlier (known at the origin)
        exog_future = pd.DataFrame(
            series.loc[f"{year - 1}-01-01":f"{year - 1}-12-01", EXOG_COLS].to_numpy(),
            index=future, columns=EXOG_COLS
        )

        forecast = result.get_forecast(steps=HORIZON, exog=exog_future).predicted_mean

        forecasts.append(pd.DataFrame({
            "component":     "sarimax_csi",
            "year":          year,
            "grid_number":   grid_number,
            "target_date":   future,
            "predicted_csi": forecast.to_numpy()
        }))

        diagnostics.append({
            "year":              year,
            "grid_number":       grid_number,
            "order":             str(order),
            "seasonal_order":    str(seasonal_order),
            "train_end":         f"{train_end:%Y-%m}",
            "aic":               result.aic,
            "converged":         bool(result.mle_retvals.get("converged", True)),
            "ljung_box_p_lag12": acorr_ljungbox(
                result.resid.iloc[order[1] + SEASONAL_D * SEASONAL_PERIOD:], lags=[SEASONAL_PERIOD]
            )["lb_pvalue"].iloc[0]
        })

    return forecasts, diagnostics


print("\n" + "=" * 65)
print(f"SARIMAX (CSI) COMPONENT: forecasts for {FORECAST_YEARS}")
print("=" * 65)

outputs = Parallel(n_jobs=N_JOBS, verbose=0)(
    delayed(run_sarimax_grid)(grid_number, frame, *SARIMAX_ORDERS[grid_number])
    for grid_number, frame in data.groupby("grid_number")
)

sarimax_forecasts = pd.concat([table for output in outputs for table in output[0]], ignore_index=True)

sarimax_diagnostics = pd.DataFrame([row for output in outputs for row in output[1]])

sarimax_diagnostics.to_csv(MODEL_DIR / "sarimax_csi_diagnostics.csv", index=False)

print(f"Fits: {len(sarimax_diagnostics)}   not converged: {(~sarimax_diagnostics['converged']).sum()}")

## LSTM (CSI + penalized loss) component

Same architecture, loss and seed as in model development. The forecast input is the 12-month window ending at the December origin.

In [ ]:
# ============================================================
# 7. LSTM (CSI + PENALIZED LOSS) COMPONENT
# ============================================================

import torch
from torch import nn
from torch.utils.data import DataLoader, TensorDataset
from sklearn.preprocessing import MinMaxScaler


# Settings identical to model development
HIDDEN_SIZE = 64

NUM_LAYERS = 2

DROPOUT = 0.2

LEARNING_RATE = 1e-3

BATCH_SIZE = 64

MAX_EPOCHS = 200

PATIENCE = 20

PENALTY_SETTINGS = {
    "csi_max":          CSI_MAX,
    "tolerance":        0.05,
    "violation_weight": 10.0,
    "error_weight":     1.0,
    "reward_weight":    0.005
}

INPUTS = [CSI_COL, CLOUD_COL, RH_COL]

SCALED_COLS = INPUTS + STATIC_COLS

FEATURES = SCALED_COLS + CALENDAR_COLS

torch.use_deterministic_algorithms(True)


class PenalizedClearSkyLoss(nn.Module):
    """MSE + clear-sky violation penalty + tolerance penalty - bounded accuracy reward (on CSI)."""

    def __init__(self, csi_max, tolerance, violation_weight, error_weight, reward_weight):

        super().__init__()

        self.csi_max = csi_max
        self.tolerance = tolerance
        self.violation_weight = violation_weight
        self.error_weight = error_weight
        self.reward_weight = reward_weight

    def forward(self, prediction, target):

        error = prediction - target

        mse = (error ** 2).mean()

        violation = (torch.relu(prediction - self.csi_max) ** 2 + torch.relu(-prediction) ** 2).mean()

        excess_error = (torch.relu(error.abs() - self.tolerance) ** 2).mean()

        physical = ((prediction >= 0) & (prediction <= self.csi_max)).float()

        reward = (torch.exp(-error ** 2 / (2 * self.tolerance ** 2)) * physical).mean()

        return (mse + self.violation_weight * violation + self.error_weight * excess_error
                - self.reward_weight * reward)


class LSTMForecaster(nn.Module):
    """Stacked LSTM with a multi-output linear head."""

    def __init__(self, n_features, hidden_size, num_layers, dropout, horizon):

        super().__init__()

        self.lstm = nn.LSTM(input_size=n_features, hidden_size=hidden_size, num_layers=num_layers,
                            dropout=dropout if num_layers > 1 else 0.0, batch_first=True)

        self.dropout = nn.Dropout(dropout)

        self.output = nn.Linear(hidden_size, horizon)

    def forward(self, x):

        sequence_output, _ = self.lstm(x)

        return self.output(self.dropout(sequence_output[:, -1, :]))


def make_sequences(frame, feature_cols, target_col, window, horizon):
    """X = 12 months up to each origin; Y = the next 12 months (training samples)."""

    X, Y, grid_numbers, origins = [], [], [], []

    for grid_number, group in frame.groupby("grid_number"):

        features = group[feature_cols].to_numpy(dtype=np.float32)

        target = group[target_col].to_numpy(dtype=np.float32)

        dates = group["date"].to_numpy()

        for i in range(window - 1, len(group) - horizon):

            X.append(features[i - window + 1:i + 1])
            Y.append(target[i + 1:i + 1 + horizon])
            grid_numbers.append(grid_number)
            origins.append(dates[i])

    return np.stack(X), np.stack(Y), pd.DataFrame({"grid_number": grid_numbers,
                                                   "origin_date": pd.to_datetime(origins)})


def train_lstm(X_fit, Y_fit, X_es, Y_es, loss_function):
    """Train with early stopping on the model's own loss."""

    set_seed(SEED)

    torch.manual_seed(SEED)

    model = LSTMForecaster(X_fit.shape[2], HIDDEN_SIZE, NUM_LAYERS, DROPOUT, HORIZON)

    optimizer = torch.optim.Adam(model.parameters(), lr=LEARNING_RATE)

    loader = DataLoader(TensorDataset(torch.from_numpy(X_fit), torch.from_numpy(Y_fit)),
                        batch_size=BATCH_SIZE, shuffle=True,
                        generator=torch.Generator().manual_seed(SEED))

    X_es, Y_es = torch.from_numpy(X_es), torch.from_numpy(Y_es)

    best_loss, best_state, best_epoch, wait = np.inf, None, 0, 0

    for epoch in range(1, MAX_EPOCHS + 1):

        model.train()

        for X_batch, Y_batch in loader:

            optimizer.zero_grad()

            loss = loss_function(model(X_batch), Y_batch)

            loss.backward()

            optimizer.step()

        model.eval()

        with torch.no_grad():
            es_loss = loss_function(model(X_es), Y_es).item()

        if es_loss < best_loss:
            best_loss, best_state, best_epoch, wait = es_loss, copy.deepcopy(model.state_dict()), epoch, 0
        else:
            wait += 1
            if wait >= PATIENCE:
                break

    model.load_state_dict(best_state)

    model.eval()

    return model, best_epoch, best_loss


print("\n" + "=" * 65)
print(f"LSTM (CSI + PL) COMPONENT: forecasts for {FORECAST_YEARS}")
print("=" * 65)

lstm_tables = []

for year in FORECAST_YEARS:

    train_end = pd.Timestamp(f"{year - 1}-12-01")

    frame = data[data["date"] <= train_end].copy()

    # Scaler fitted on the training rows only
    scaler = MinMaxScaler().fit(frame[SCALED_COLS])

    scaled = frame.copy()

    scaled[SCALED_COLS] = scaler.transform(frame[SCALED_COLS])

    scaled["__target__"] = frame[CSI_COL]

    X, Y, keys = make_sequences(scaled, FEATURES, "__target__", WINDOW, HORIZON)

    last_target = keys["origin_date"] + pd.DateOffset(months=HORIZON)

    early_stop_start = train_end - pd.DateOffset(months=11)

    fit_mask = (last_target < early_stop_start).to_numpy()

    es_mask = ((last_target >= early_stop_start) & (last_target <= train_end)).to_numpy()

    model, best_epoch, best_loss = train_lstm(
        X[fit_mask], Y[fit_mask], X[es_mask], Y[es_mask], PenalizedClearSkyLoss(**PENALTY_SETTINGS)
    )

    # Forecast input: the 12 months ending at the December origin
    groups = list(scaled.groupby("grid_number"))

    X_origin = np.stack([group[FEATURES].to_numpy(dtype=np.float32)[-WINDOW:] for _, group in groups])

    with torch.no_grad():
        predicted = model(torch.from_numpy(X_origin)).numpy()

    grids = np.array([grid_number for grid_number, _ in groups])

    lstm_tables.append(pd.DataFrame({
        "component":     "lstm_csi_pl",
        "year":          year,
        "grid_number":   np.repeat(grids, HORIZON),
        "target_date":   np.tile(pd.date_range(f"{year}-01-01", periods=HORIZON, freq="MS"), len(grids)),
        "predicted_csi": predicted.ravel()
    }))

    torch.save(
        {"state_dict": model.state_dict(), "features": FEATURES, "scaled_cols": SCALED_COLS,
         "scaler_min": scaler.data_min_.tolist(), "scaler_max": scaler.data_max_.tolist(),
         "penalty_settings": PENALTY_SETTINGS, "train_end": f"{train_end:%Y-%m}",
         "forecast_year": year, "best_epoch": best_epoch, "seed": SEED},
        MODEL_DIR / f"lstm_csi_pl_{year}.pt"
    )

    print(f"Forecast {year}: trained on {FIRST_DATA_YEAR}-{year - 1}   fit {fit_mask.sum()}  "
          f"early-stop {es_mask.sum()}  best epoch {best_epoch}  early-stop loss {best_loss:.5f}")

lstm_forecasts = pd.concat(lstm_tables, ignore_index=True)

## Combination, check year and final forecast

The TVWA weights use every earlier out-of-sample component forecast (2014 onward), discounted by δ = 0.8 per year. The check year (2025) is forecast from December 2024 and compared with the observations, including climatology as a reference.

In [ ]:
# ============================================================
# 8. TVWA COMBINATION, CHECK-YEAR ACCURACY, FINAL FORECAST
# ============================================================

DISCOUNT = 0.8

CLEARSKY_LOOKUP = clearsky[["grid_number", "date", CLEAR_COL]].rename(columns={"date": "target_date"})

ACTUALS = data[["grid_number", "date", GHI_COL]].rename(columns={"date": "target_date", GHI_COL: "actual"})


# ------------------------------------------------------------
# New component forecasts in GHI (CSI x clear-sky GHI)
# ------------------------------------------------------------

new_components = (
    pd.concat([sarimax_forecasts, lstm_forecasts], ignore_index=True)
    .merge(CLEARSKY_LOOKUP, on=["grid_number", "target_date"], how="left")
)

new_components["predicted_ghi"] = new_components["predicted_csi"] * new_components[CLEAR_COL]


# ------------------------------------------------------------
# Error history: development forecasts before the new years
# ------------------------------------------------------------

history = pd.concat(
    [pd.read_csv(path, parse_dates=["target_date"]) for paths in COMPONENT_HISTORY.values() for path in paths],
    ignore_index=True
)

history = history[history["year"] < min(FORECAST_YEARS)]

columns = ["component", "year", "grid_number", "target_date", "predicted_ghi"]

pair = (
    pd.concat([history[columns], new_components[columns]], ignore_index=True)
    .pivot_table(index=["year", "grid_number", "target_date"], columns="component",
                 values="predicted_ghi")
    .reset_index()
    .merge(ACTUALS, on=["grid_number", "target_date"], how="left")
    .rename(columns={"sarimax_csi": "sarimax", "lstm_csi_pl": "lstm"})
)

print(f"Component forecast years: {sorted(pair['year'].unique())}")


# ------------------------------------------------------------
# TVWA weights (discounted inverse-MSE per grid) and combination
# ------------------------------------------------------------

combined, weight_tables = [], []

for year in FORECAST_YEARS:

    past = pair[(pair["year"] < year) & pair["actual"].notna()]

    discount = DISCOUNT ** (year - 1 - past["year"])

    discounted = (
        past.assign(
            d_sarimax=discount * (past["sarimax"] - past["actual"]) ** 2,
            d_lstm=discount * (past["lstm"] - past["actual"]) ** 2
        )
        .groupby("grid_number")[["d_sarimax", "d_lstm"]].sum()
    )

    w_sarimax = (discounted["d_lstm"] / (discounted["d_sarimax"] + discounted["d_lstm"])).fillna(0.5)

    weight_tables.append(pd.DataFrame({
        "year": year, "grid_number": w_sarimax.index, "w_sarimax": w_sarimax.to_numpy(),
        "w_lstm": 1 - w_sarimax.to_numpy(),
        "history_years": f"{past['year'].min()}-{past['year'].max()}"
    }))

    current = pair[pair["year"] == year].copy()

    current["w_sarimax"] = current["grid_number"].map(w_sarimax)

    current["predicted"] = current["w_sarimax"] * current["sarimax"] + (1 - current["w_sarimax"]) * current["lstm"]

    combined.append(current)

proposed = pd.concat(combined, ignore_index=True).merge(
    CLEARSKY_LOOKUP, on=["grid_number", "target_date"], how="left"
)

weights = pd.concat(weight_tables, ignore_index=True)

weights.to_csv(MODEL_DIR / "tvwa_weights.csv", index=False)

for year, table in weights.groupby("year"):
    print(f"TVWA {year}: history {table['history_years'].iloc[0]}, "
          f"mean SARIMAX weight {table['w_sarimax'].mean():.2f}")


# ------------------------------------------------------------
# Check year: accuracy against the observations
# ------------------------------------------------------------

def pooled(actual, predicted):

    error = predicted - actual

    rmse = np.sqrt(np.mean(error ** 2))

    return {
        "RMSE": rmse,
        "MAE": np.mean(np.abs(error)),
        "nRMSE_pct": rmse / np.mean(actual) * 100,
        "R2": 1 - np.sum(error ** 2) / np.sum((actual - np.mean(actual)) ** 2),
        "n_forecasts": len(actual)
    }


check_frame = proposed[proposed["year"] == CHECK_YEAR].copy()

monthly_mean = (
    data[data["year"] < CHECK_YEAR].groupby(["grid_number", "month"], as_index=False)[GHI_COL].mean()
    .rename(columns={GHI_COL: "climatology"})
)

check_frame = check_frame.assign(month=check_frame["target_date"].dt.month).merge(
    monthly_mean, on=["grid_number", "month"], how="left"
)

check_metrics = pd.DataFrame({
    f"{PROPOSED} (proposed)":    pooled(check_frame["actual"], check_frame["predicted"]),
    "SARIMAX (CSI) component":   pooled(check_frame["actual"], check_frame["sarimax"]),
    "LSTM (CSI + PL) component": pooled(check_frame["actual"], check_frame["lstm"]),
    "Climatology":               pooled(check_frame["actual"], check_frame["climatology"])
}).T

check_metrics["skill_vs_climatology"] = 1 - check_metrics["RMSE"] / check_metrics.loc["Climatology", "RMSE"]

check_metrics.index.name = "model"

check_metrics.to_csv(OUTPUT_DIR / f"check_{CHECK_YEAR}_metrics.csv")

check_frame.to_csv(OUTPUT_DIR / f"check_{CHECK_YEAR}_predictions.csv", index=False)

print("\n" + "=" * 65)
print(f"CHECK YEAR {CHECK_YEAR}: forecast from Dec {CHECK_YEAR - 1} vs observations")
print("=" * 65)

print(check_metrics.round(4).to_string())


# ------------------------------------------------------------
# Final forecast: monthly table
# ------------------------------------------------------------

forecast = (
    proposed[proposed["year"] == FORECAST_YEAR]
    .merge(GRID_INFO, on="grid_number", how="left")
    .rename(columns={"target_date": "date", "predicted": "ghi_forecast_kwh_m2_day",
                     "sarimax": "sarimax_component_ghi", "lstm": "lstm_component_ghi",
                     CLEAR_COL: "clear_sky_ghi_kwh_m2_day"})
)

forecast["month"] = forecast["date"].dt.month

forecast["csi_forecast"] = forecast["ghi_forecast_kwh_m2_day"] / forecast["clear_sky_ghi_kwh_m2_day"]

forecast = forecast[
    ["grid_number", "grid_id", "latitude", "longitude", "date", "month", "ghi_forecast_kwh_m2_day",
     "csi_forecast", "clear_sky_ghi_kwh_m2_day", "sarimax_component_ghi", "lstm_component_ghi", "w_sarimax"]
].sort_values(["grid_number", "date"]).reset_index(drop=True)

if len(forecast) != N_GRIDS * HORIZON or forecast["ghi_forecast_kwh_m2_day"].isna().any():
    raise ValueError("The final forecast is incomplete")

forecast.to_csv(OUTPUT_DIR / f"forecast_{FORECAST_YEAR}_monthly.csv", index=False)

violations = int((forecast["csi_forecast"] > CSI_MAX).sum())


# ------------------------------------------------------------
# Final forecast: annual summary per grid
# ------------------------------------------------------------

def annual_summary(frame, value):
    """Annual total (kWh/m²) and day-weighted annual mean daily GHI per grid and year."""

    days = frame["date"].dt.days_in_month

    totals = (frame.assign(total=frame[value] * days, days=days)
              .groupby(["grid_number", frame["date"].dt.year])[["total", "days"]].sum())

    totals["mean_daily"] = totals["total"] / totals["days"]

    return totals


forecast_annual = annual_summary(forecast, "ghi_forecast_kwh_m2_day").reset_index(level=1, drop=True)

observed_annual = annual_summary(data, GHI_COL)

baseline = observed_annual.groupby("grid_number")["mean_daily"].mean()

annual = GRID_INFO.set_index("grid_number").join(pd.DataFrame({
    "annual_mean_daily_ghi_kwh_m2_day": forecast_annual["mean_daily"],
    "annual_total_ghi_kwh_m2": forecast_annual["total"],
    f"mean_daily_ghi_{FIRST_DATA_YEAR}_{LAST_DATA_YEAR}": baseline
}))

annual["difference_from_average_pct"] = (
    (annual["annual_mean_daily_ghi_kwh_m2_day"] / annual[f"mean_daily_ghi_{FIRST_DATA_YEAR}_{LAST_DATA_YEAR}"] - 1) * 100
)

if PER_GRID_TESTS_FILE.exists():

    per_grid = pd.read_csv(PER_GRID_TESTS_FILE)

    typical = per_grid[per_grid["model_A"].str.startswith(PROPOSED)].groupby("grid_number")["RMSE_A"].first()

    annual["typical_monthly_rmse_2015_2024"] = typical

annual = annual.reset_index()

annual.to_csv(OUTPUT_DIR / f"forecast_{FORECAST_YEAR}_annual.csv", index=False)


print("\n" + "=" * 65)
print(f"FORECAST {FORECAST_YEAR} ({PROPOSED}, trained on {FIRST_DATA_YEAR}-{LAST_DATA_YEAR})")
print("=" * 65)

national = forecast.groupby("month")["ghi_forecast_kwh_m2_day"].mean()

print(f"Annual mean daily GHI, mean of grids: {annual['annual_mean_daily_ghi_kwh_m2_day'].mean():.3f} {UNITS} "
      f"(range {annual['annual_mean_daily_ghi_kwh_m2_day'].min():.3f}-"
      f"{annual['annual_mean_daily_ghi_kwh_m2_day'].max():.3f})")

print(f"Annual total, mean of grids: {annual['annual_total_ghi_kwh_m2'].mean():.0f} kWh/m²")

print(f"Difference from the {FIRST_DATA_YEAR}-{LAST_DATA_YEAR} average: "
      f"{annual['difference_from_average_pct'].mean():+.2f}% (mean of grids)")

print(f"Sunniest month: {pd.Timestamp(2000, national.idxmax(), 1):%B} ({national.max():.2f}); "
      f"least sunny: {pd.Timestamp(2000, national.idxmin(), 1):%B} ({national.min():.2f})")

print(f"Clear-sky violations (forecast GHI > clear-sky GHI): {violations} of {len(forecast)}")

print(f"\nSaved: {OUTPUT_DIR / f'forecast_{FORECAST_YEAR}_monthly.csv'}")
print(f"Saved: {OUTPUT_DIR / f'forecast_{FORECAST_YEAR}_annual.csv'}")
print(f"Saved: {OUTPUT_DIR / f'check_{CHECK_YEAR}_metrics.csv'}")

## Maps of the final forecast

Orange shades encode GHI (darker = more sunshine). The blue shades of the error maps stay reserved for errors. All 12 monthly maps share one colour scale.

In [ ]:
# ============================================================
# 9. MAPS OF THE FINAL FORECAST
# ============================================================

import geopandas as gpd
import matplotlib.pyplot as plt
from matplotlib.colors import LinearSegmentedColormap, Normalize
from matplotlib.cm import ScalarMappable


INK, INK_2, AXIS = "#0b0b0b", "#52514e", "#c3c2b7"

# Sequential orange ramp (light = less sunshine, dark = more), lightness-matched
# to the blue ramp used for the error maps
ORANGE = ["#fad7c9", "#f7c3ae", "#f4af93", "#f19875", "#ee8156", "#eb6834", "#d45e2f",
          "#bf542a", "#a94b25", "#944121", "#80381c", "#6b3018", "#582714"]

GHI_CMAP = LinearSegmentedColormap.from_list("ghi_orange", ORANGE)

plt.rcParams.update({
    "font.family": "Arial", "font.size": 7.5,
    "xtick.color": "#898781", "ytick.color": "#898781",
    "xtick.labelcolor": INK_2, "ytick.labelcolor": INK_2,
    "xtick.major.size": 2.5, "ytick.major.size": 2.5,
    "xtick.major.width": 0.6, "ytick.major.width": 0.6,
    "svg.fonttype": "none", "pdf.fonttype": 42
})

grid_shapes = gpd.read_file(GRID_GEOJSON)

# Simplify the detailed coastline (about 13 million vertices) before reprojecting,
# which keeps memory use low: ~500 m tolerance (0.005° if the file is in degrees)
boundary = gpd.read_file(BOUNDARY_FILE)

boundary["geometry"] = boundary.geometry.simplify(0.005 if boundary.crs.is_geographic else 500)

boundary = boundary.to_crs(grid_shapes.crs)

LON_LIMITS, LAT_LIMITS = (115.8, 127.2), (3.8, 22.2)

MAP_ASPECT = (LAT_LIMITS[1] - LAT_LIMITS[0]) / (LON_LIMITS[1] - LON_LIMITS[0])


def add_axes_inches(fig, left, top, width, height):
    """Axes positioned in inches from the figure's top-left corner."""

    fig_width, fig_height = fig.get_size_inches()

    return fig.add_axes([left / fig_width, (fig_height - top - height) / fig_height,
                         width / fig_width, height / fig_height])


def draw_map(ax, values, column, norm, show_x=True, show_y=True, ticks=True):

    shapes = grid_shapes.merge(values, on="grid_number", how="left")

    shapes.plot(ax=ax, column=column, cmap=GHI_CMAP, norm=norm, edgecolor="white", linewidth=0.5, zorder=1)

    boundary.plot(ax=ax, facecolor="none", edgecolor=INK_2, linewidth=0.3, zorder=2)

    ax.set_xlim(*LON_LIMITS)
    ax.set_ylim(*LAT_LIMITS)
    ax.set_aspect("equal")
    ax.set_xlabel("")
    ax.set_ylabel("")

    if ticks:
        ax.set_xticks([118, 122, 126])
        ax.set_yticks([6, 10, 14, 18, 22])
        ax.set_xticklabels([f"{t}°E" for t in [118, 122, 126]])
        ax.set_yticklabels([f"{t}°N" for t in [6, 10, 14, 18, 22]])
        ax.tick_params(labelsize=6.5, labelbottom=show_x, labelleft=show_y)
    else:
        ax.set_xticks([])
        ax.set_yticks([])

    for side in ax.spines.values():
        side.set_color(AXIS)
        side.set_linewidth(0.6)


def save(fig, name):

    for extension in ("png", "svg", "pdf"):
        fig.savefig(FIGURE_DIR / f"{name}.{extension}", dpi=300, bbox_inches="tight",
                    pad_inches=0.04, facecolor="white")

    print(f"Saved: {FIGURE_DIR / name}.png / .svg / .pdf")


# ------------------------------------------------------------
# Map 1: annual GHI, colour bar in two units
#   right: annual mean of daily GHI (kWh/m²/day)
#   left:  annual total GHI (kWh/m²/year) = mean daily x days in the year
# ------------------------------------------------------------

DAYS_IN_YEAR = pd.Timestamp(FORECAST_YEAR, 12, 31).dayofyear

annual_values = annual[["grid_number", "annual_mean_daily_ghi_kwh_m2_day"]]

low = np.floor(annual_values.iloc[:, 1].min() * 10) / 10

high = np.ceil(annual_values.iloc[:, 1].max() * 10) / 10

annual_norm = Normalize(vmin=low, vmax=high)

FIG_W, PANEL_W = 5.0, 3.0

PANEL_H = PANEL_W * MAP_ASPECT

fig = plt.figure(figsize=(FIG_W, 0.05 + 0.62 + PANEL_H + 0.3))

fig_h = fig.get_size_inches()[1]

fig.text(0.02 / FIG_W, 1 - 0.05 / fig_h, f"Forecast GHI for {FORECAST_YEAR}",
         fontsize=9, fontweight="bold", color=INK, va="top", ha="left")

fig.text(0.02 / FIG_W, 1 - 0.27 / fig_h,
         f"Annual GHI per grid cell (mean daily value and annual total) · {PROPOSED},\n"
         f"trained on {FIRST_DATA_YEAR}–{LAST_DATA_YEAR}",
         fontsize=7, color=INK_2, va="top", ha="left", linespacing=1.3)

ax = add_axes_inches(fig, 0.45, 0.67, PANEL_W, PANEL_H)

draw_map(ax, annual_values, "annual_mean_daily_ghi_kwh_m2_day", annual_norm)

# Room on the left of the colour bar for the second set of labels
colorbar_ax = add_axes_inches(fig, 0.45 + PANEL_W + 0.8, 0.67 + 0.6, 0.11, PANEL_H - 1.2)

colorbar = fig.colorbar(ScalarMappable(norm=annual_norm, cmap=GHI_CMAP), cax=colorbar_ax)

colorbar.set_label(f"Annual mean of daily GHI ({UNITS})", fontsize=7, color=INK_2)

colorbar.outline.set_visible(False)

colorbar.ax.tick_params(labelsize=6.5, length=2)

# The same scale expressed as annual totals (an exact unit conversion)
totals = colorbar.ax.secondary_yaxis(
    "left", functions=(lambda x: x * DAYS_IN_YEAR, lambda x: x / DAYS_IN_YEAR)
)

step = 100 if (high - low) * DAYS_IN_YEAR > 300 else 50

total_ticks = np.arange(np.ceil(low * DAYS_IN_YEAR / step) * step, high * DAYS_IN_YEAR + 1e-6, step)

totals.set_yticks(total_ticks)

totals.set_yticklabels([f"{tick:,.0f}" for tick in total_ticks])

totals.tick_params(labelsize=6.5, length=2)

totals.spines["left"].set_visible(False)

totals.set_ylabel("Annual total GHI (kWh/m²/year)", fontsize=7, color=INK_2)

save(fig, f"forecast_{FORECAST_YEAR}_annual_map")

plt.show()


# ------------------------------------------------------------
# Map 2: the 12 monthly forecasts (shared colour scale)
# ------------------------------------------------------------

values = forecast["ghi_forecast_kwh_m2_day"]

monthly_norm = Normalize(vmin=np.floor(values.min() * 2) / 2, vmax=np.ceil(values.max() * 2) / 2)

FIG_W, LEFT, PANEL_W, GUTTER = 6.5, 0.42, 1.38, 0.12

PANEL_H = PANEL_W * MAP_ASPECT

HEADING, TITLE_BAND, ROW_GAP, BOTTOM = 0.32, 0.2, 0.1, 0.62

FIG_H = 0.05 + HEADING + 3 * (TITLE_BAND + PANEL_H) + 2 * ROW_GAP + BOTTOM

fig = plt.figure(figsize=(FIG_W, FIG_H))

fig.text(0.02 / FIG_W, 1 - 0.05 / FIG_H,
         f"Forecast monthly GHI for {FORECAST_YEAR} · {PROPOSED}",
         fontsize=8.5, fontweight="bold", color=INK, va="top", ha="left")

for month in range(1, 13):

    row, column = divmod(month - 1, 4)

    left = LEFT + column * (PANEL_W + GUTTER)

    top = 0.05 + HEADING + row * (TITLE_BAND + PANEL_H + ROW_GAP) + TITLE_BAND

    ax = add_axes_inches(fig, left, top, PANEL_W, PANEL_H)

    month_values = forecast.loc[forecast["month"] == month, ["grid_number", "ghi_forecast_kwh_m2_day"]]

    draw_map(ax, month_values, "ghi_forecast_kwh_m2_day", monthly_norm,
             show_x=row == 2, show_y=column == 0)

    ax.text(0.0, 1.0 + 0.05 / PANEL_H,
            f"{pd.Timestamp(FORECAST_YEAR, month, 1):%b}  ·  mean {month_values.iloc[:, 1].mean():.2f}",
            transform=ax.transAxes, fontsize=7.2, fontweight="bold", color=INK, ha="left", va="bottom")

colorbar_ax = add_axes_inches(fig, 1.5, FIG_H - 0.36, 3.5, 0.1)

colorbar = fig.colorbar(ScalarMappable(norm=monthly_norm, cmap=GHI_CMAP), cax=colorbar_ax,
                        orientation="horizontal")

colorbar.set_label(f"Forecast monthly mean of daily GHI ({UNITS})", fontsize=7, color=INK_2)

colorbar.outline.set_visible(False)

colorbar.ax.tick_params(labelsize=6.5, length=2)

save(fig, f"forecast_{FORECAST_YEAR}_monthly_maps")

plt.show()

## Data acknowledgement and references

**NASA POWER data:** *"The data was obtained from the National Aeronautics and Space Administration (NASA) Langley Research Center (LaRC) Prediction of Worldwide Energy Resource (POWER) Project funded through the NASA Earth Science/Applied Science Program."* https://power.larc.nasa.gov

The models, combination and clear-sky method are those of `Hybrid_Models.ipynb`; see its reference list (statsmodels, PyTorch, pvlib, Ineichen & Perez 2002, Bates & Granger 1969, Stock & Watson 2004, among others). Extraterrestrial irradiation for the validation check follows Duffie, J. A., & Beckman, W. A. (2013). *Solar engineering of thermal processes* (4th ed.). Wiley.